In [3]:
import pandas as pd
import numpy as np
import ccxt
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import polars as pl
import os

In [4]:
df_23 = pd.read_csv("BTCUSDT_5m_2023.csv")
df_24 = pd.read_csv("BTCUSDT_5m_2024.csv")
df_25 = pd.read_csv("BTCUSDT_5m_2025.csv")

In [5]:
df = pd.concat([df_23, df_24, df_25], ignore_index=True)

In [6]:
print(df.duplicated().sum())

2


In [8]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


In [11]:
df.set_index("Open Time", inplace=True)

In [9]:
df = df.drop("Ignore", axis=1)

In [10]:
df.shape

(315633, 11)

In [19]:
# Verify Time Continuity

df.index = pd.to_datetime(df.index)
time_diff = df.index.to_series().diff()

print(time_diff.value_counts().head())

Open Time
0 days 00:05:00    315631
0 days 01:25:00         1
Name: count, dtype: int64


In [20]:
time_diff = df.index.to_series().diff()

gap = time_diff[time_diff != pd.Timedelta(minutes=5)]

print(gap)

Open Time
2022-12-31 18:30:00               NaT
2023-03-24 14:00:00   0 days 01:25:00
Name: Open Time, dtype: timedelta64[us]


In [21]:
gap_time = pd.Timestamp("2023-03-24 14:00:00")

location = df.index.get_loc(gap_time)

df.iloc[location-3:location+3]

,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume
Open Time,,,,,,,,,,
2023-03-24 12:25:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:29:59.999,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 12:30:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:34:59.999,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 12:35:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:39:41.646,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 14:00:00,28079.99,28079.99,27835.00,27858.24,1209.62045,2023-03-24 14:04:59.999,3.374709e+07,24077,346.49241,9.669154e+06
2023-03-24 14:05:00,27858.23,27950.00,27857.58,27916.47,571.88072,2023-03-24 14:09:59.999,1.596052e+07,10546,306.90519,8.565686e+06
2023-03-24 14:10:00,27916.47,28253.01,27911.97,28160.01,1169.30835,2023-03-24 14:14:59.999,3.283858e+07,20664,737.35329,2.070980e+07


In [ ]:
df[df["Number of Trades"] == 0]

,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume
Open Time,,,,,,,,,,
2023-03-24 11:30:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:34:59.999,0.0,0,0.0,0.0
2023-03-24 11:35:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:39:59.999,0.0,0,0.0,0.0
2023-03-24 11:40:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:44:59.999,0.0,0,0.0,0.0
2023-03-24 11:45:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:49:59.999,0.0,0,0.0,0.0
2023-03-24 11:50:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:54:59.999,0.0,0,0.0,0.0
2023-03-24 11:55:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:59:59.999,0.0,0,0.0,0.0
2023-03-24 12:00:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:04:59.999,0.0,0,0.0,0.0
2023-03-24 12:05:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:09:59.999,0.0,0,0.0,0.0
2023-03-24 12:10:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:14:59.999,0.0,0,0.0,0.0


In [25]:
expected = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="5min"
)

missing = expected.difference(df.index)

print(missing)

DatetimeIndex(['2023-03-24 12:40:00', '2023-03-24 12:45:00',
               '2023-03-24 12:50:00', '2023-03-24 12:55:00',
               '2023-03-24 13:00:00', '2023-03-24 13:05:00',
               '2023-03-24 13:10:00', '2023-03-24 13:15:00',
               '2023-03-24 13:20:00', '2023-03-24 13:25:00',
               '2023-03-24 13:30:00', '2023-03-24 13:35:00',
               '2023-03-24 13:40:00', '2023-03-24 13:45:00',
               '2023-03-24 13:50:00', '2023-03-24 13:55:00'],
              dtype='datetime64[us]', freq='5min')


In [ ]:
exchange = ccxt.bybit()

ohlcv = exchange.fetch_ohlcv(
    'BTC/USDT',
    timeframe='5m',
    since=exchange.parse8601('2023-03-24T12:40:00Z'),
    limit=20
)

df = pd.DataFrame(
    ohlcv,
    columns=[
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
print("ByBit Exchange Data: ")
print()
print(df)

ByBit Exchange Data: 

             timestamp      Open      High       Low     Close      Volume
0  2023-03-24 12:40:00  27940.97  28026.15  27914.01  28016.09   62.325252
1  2023-03-24 12:45:00  28016.09  28042.02  27974.65  28006.65   81.581113
2  2023-03-24 12:50:00  28006.65  28006.65  27958.02  27964.92   20.937888
3  2023-03-24 12:55:00  27964.92  28016.11  27964.91  27969.38   19.270613
4  2023-03-24 13:00:00  27969.38  27986.37  27914.53  27914.53   33.361064
5  2023-03-24 13:05:00  27914.53  27950.00  27871.31  27922.85   39.925953
6  2023-03-24 13:10:00  27922.85  27927.95  27883.23  27884.21   23.886663
7  2023-03-24 13:15:00  27884.21  27919.96  27737.60  27779.75   95.644659
8  2023-03-24 13:20:00  27779.75  27832.89  27706.13  27800.55   87.467372
9  2023-03-24 13:25:00  27800.55  27921.27  27783.95  27898.33   34.431304
10 2023-03-24 13:30:00  27898.33  27921.26  27845.89  27874.54   24.871217
11 2023-03-24 13:35:00  27874.54  27893.05  27786.13  27786.13   36.389894
12

In [2]:

exchange = ccxt.coinbase()

ohlcv = exchange.fetch_ohlcv(
    'BTC/USDT',
    timeframe='5m',
    since=exchange.parse8601('2023-03-24T12:40:00Z'),
    limit=20
)

df = pd.DataFrame(
    ohlcv,
    columns=[
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
print("Coinbase Exchange Data: ")
print()
print(df)

Coinbase Exchange Data: 

             timestamp      Open      High       Low     Close     Volume
0  2023-03-24 12:40:00  27932.02  28019.93  27921.60  27997.09  10.958905
1  2023-03-24 12:45:00  28006.18  28038.27  27990.06  28007.41   7.048360
2  2023-03-24 12:50:00  27996.21  27996.39  27971.09  27971.09   1.861593
3  2023-03-24 12:55:00  27970.23  28010.75  27970.23  27972.49   4.759529
4  2023-03-24 13:00:00  27953.69  27984.93  27926.70  27926.70   2.454821
5  2023-03-24 13:05:00  27926.70  27956.45  27890.23  27918.46   2.717892
6  2023-03-24 13:10:00  27915.64  27915.64  27840.71  27840.71   2.959891
7  2023-03-24 13:15:00  27856.54  27926.57  27744.82  27765.76   4.004068
8  2023-03-24 13:20:00  27781.86  27839.16  27709.00  27795.23  10.060609
9  2023-03-24 13:25:00  27805.94  27934.70  27805.94  27904.64  10.629298
10 2023-03-24 13:30:00  27889.91  27919.37  27843.80  27875.36   8.028627
11 2023-03-24 13:35:00  27878.99  27898.73  27784.53  27789.68   5.201914
12 2023-03-2

In [ ]:
desktop_path = os.path.expanduser("~/Desktop/BTCUSDT_5m_2023-2025_features_compressed.parquet")
df = pd.read_parquet(desktop_path)
print(df.shape)

(315633, 375)


In [ ]:
df.info(verbose=True, show_counts=True)

In [14]:
print(df.duplicated().sum())

0


In [15]:
df.head()

,Open Time,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume,Ignore,hl2,hlc3,ohlc4,typical_price,median_price,price_change,high_low_diff,close_open_diff,high_close_diff,close_low_diff,return_1,return_3,return_6,return_12,return_24,log_return_1,log_return_3,close_open_ratio,high_low_ratio,close_high_ratio,close_low_ratio,price_position,gap,gap_pct,dist_high,dist_low,dist_open,body,body_signed,bullish,bearish,doji,range,upper_wick,lower_wick,body_ratio,upper_wick_ratio,lower_wick_ratio,close_position,open_position,upper_wick_dominant,lower_wick_dominant,marubozu,hammer,shooting_star,spinning_top,year,quarter,month,week,day,day_of_week,day_of_year,hour,minute,is_weekend,month_start,month_end,quarter_start,quarter_end,asia_session,london_session,newyork_session,london_ny_overlap,session,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,asia_open,london_open,ny_open,minutes_since_midnight,ema_10,ema_20,ema_50,ema_100,ema_200,sma_20,sma_50,sma_200,ema_dist_10,ema_dist_pct_10,ema_slope_10,ema_dist_20,ema_dist_pct_20,ema_slope_20,ema_dist_50,ema_dist_pct_50,ema_slope_50,ema_dist_100,ema_dist_pct_100,ema_slope_100,ema_dist_200,ema_dist_pct_200,ema_slope_200,sma_dist_20,sma_slope_20,sma_dist_50,sma_slope_50,sma_dist_200,sma_slope_200,ema_spread_10_20,ema_spread_pct_10_20,ema_spread_20_50,ema_spread_pct_20_50,ema_spread_50_100,ema_spread_pct_50_100,ema_spread_50_200,ema_spread_pct_50_200,ema_spread_100_200,ema_spread_pct_100_200,price_above_ema_10,price_above_ema_20,price_above_ema_50,price_above_ema_100,price_above_ema_200,bullish_alignment,bearish_alignment,adx,plus_di,minus_di,strong_trend,aroon_up,aroon_down,rsi,rsi_overbought,rsi_oversold,rsi_above_50,rsi_slope,rsi_acceleration,macd,macd_signal,macd_hist,macd_slope,macd_hist_slope,macd_above_signal,macd_cross_up,macd_cross_down,stoch_k,stoch_d,stoch_spread,stoch_cross_up,stoch_cross_down,roc,roc_slope,cci,williams_r,awesome_oscillator,tsi,tsi_slope,rsi_x_roc,macd_x_rsi,stoch_x_rsi,cci_x_roc,momentum_score,bullish_momentum,bearish_momentum,true_range,atr,atr_pct,atr_slope,historical_volatility,bb_upper,bb_middle,bb_lower,bb_width,bb_width_pct,bb_percent,dc_upper,dc_lower,dc_middle,kc_upper,kc_lower,kc_middle,rolling_std,rolling_range,volatility_expansion,atr_zscore,high_volatility,low_volatility,bb_squeeze,bb_breakout_up,bb_breakout_down,ntr,atr_ratio,bb_zscore,bb_kc_ratio,atr_acceleration,volatility_compression,realized_vol_5,realized_vol_10,realized_vol_20,realized_vol_50,vol_ratio_5_20,vol_ratio_10_50,atr_percentile,taker_sell_base_volume,taker_sell_quote_volume,buy_volume_ratio,sell_volume_ratio,buy_pressure,avg_trade_size,avg_trade_value,volume_mean_5,volume_std_5,volume_ratio_5,volume_zscore_5,volume_mean_10,volume_std_10,volume_ratio_10,volume_zscore_10,volume_mean_20,volume_std_20,volume_ratio_20,volume_zscore_20,volume_mean_50,volume_std_50,volume_ratio_50,volume_zscore_50,volume_slope,volume_acceleration,buy_pressure_slope,obv,force_index,mfi,cmf,volume_spike,buy_imbalance,sell_imbalance,relative_trade_size,quote_base_ratio,buyer_dominance,buyer_dominance_ma,volume_price_strength,volume_atr,volume_rsi,aggressive_buying,aggressive_selling,smart_money_score,rolling_high,rolling_low,breakout_up,breakout_down,higher_high,lower_low,higher_close,lower_close,hh_count,ll_count,compression,range_expansion,liquidity_sweep_high,liquidity_sweep_low,breakout_strength,pullback,swing_high,swing_low,swing_high_price,swing_low_price,is_hh,is_lh,is_hl,is_ll,last_high_label,last_low_label,bull_structure,bear_structure,bos_up,bos_down,choch_bear,choch_bull,structure_shift,fvg_bullish,fvg_bearish,fvg_size,liquidity_pool_high,liquidity_pool_low,breakout_up_confirmed,breakout_down_confirmed,trend_quality,structure_score,close_lag_1,close_diff_1,close_pct_change_1,close_lag_2,close_diff_2,close_pct_change_2,close_lag_3,close_diff_3,close_pct_change_3,close_lag_4,close_diff_4,close_pct_change_4,close_lag_5,close_diff_5,close_p

### XAU/USD (Gold/US Dollar) Analysis

In [2]:
df_21 = pd.read_csv("/Users/devangasaikia/RichGoons/Gold 2021 - 2025/XAUUSD_5m_2021.csv")
df_22 = pd.read_csv("/Users/devangasaikia/RichGoons/Gold 2021 - 2025/XAUUSD_5m_2022.csv")
df_23 = pd.read_csv("/Users/devangasaikia/RichGoons/Gold 2021 - 2025/XAUUSD_5m_2023.csv")
df_24 = pd.read_csv("/Users/devangasaikia/RichGoons/Gold 2021 - 2025/XAUUSD_5m_20241.csv")
df_25 = pd.read_csv("/Users/devangasaikia/RichGoons/Gold 2021 - 2025/XAUUSD_5m_2025.csv")

In [3]:
df_gold = pd.concat([df_21, df_22, df_23, df_24, df_25], ignore_index=True)

In [10]:
df_gold.to_excel("/Users/devangasaikia/Desktop/XAUUSD_5m_2021-2025.xlsx", index=False)

In [5]:
desktop_path = os.path.expanduser("~/Desktop/XAUUSD_5m_2021-2025.parquet")
df = pd.read_parquet(desktop_path)
print(df.shape)

(354538, 23)


In [6]:
df.head()

,Open Time,Open,High,Low,Close,Volume,atr_pct,ntr,bb_width_pct,bb_percent,bb_kc_ratio,volatility_expansion,atr_ratio,atr_zscore,bb_zscore,volatility_compression,realized_vol_5,realized_vol_10,realized_vol_20,realized_vol_50,vol_ratio_5_20,vol_ratio_10_50,atr_percentile
0,2020-12-31 18:30:00+00:00,1891.625000,1893.017944,1891.444946,1892.437988,0.15465,0.0,0.000831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-12-31 18:35:00+00:00,1892.458008,1893.458008,1892.307983,1893.328003,0.07755,0.0,0.000607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-31 18:40:00+00:00,1893.333984,1893.537964,1892.868042,1893.437988,0.05714,0.0,0.000354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-31 18:45:00+00:00,1893.368042,1893.397949,1892.927979,1893.035034,0.05101,0.0,0.000269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-31 18:50:00+00:00,1892.964966,1893.765015,1892.814941,1893.765015,0.06620,0.0,0.000502,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
